In [1]:
import os
import h5py
import wrds

WRDS_USERNAME = 'cpehlivanian'      # password is read from ~/.pgpass
START, END = '2024-01-02', '2024-12-31'
source = 'CRSP dsf_v2 (WRDS)'
filename = 'price_volume.h5'

In [2]:
def data(sym_root):
    """Daily unadjusted close price and volume for one ticker from crsp.dsf_v2."""
    conn = wrds.Connection(wrds_username=WRDS_USERNAME)
    try:
        df = conn.raw_sql(
            """
            SELECT permno, yyyymmdd, dlyprc, dlyvol
            FROM crsp.dsf_v2
            WHERE ticker = %(ticker)s
              AND dlycaldt BETWEEN %(start)s AND %(end)s
            ORDER BY dlycaldt
            """,
            params={'ticker': sym_root, 'start': START, 'end': END},
        )
    finally:
        conn.close()
    if df.empty:
        raise ValueError(f"No CRSP data for {sym_root} between {START} and {END}")
    if df['permno'].nunique() > 1:     # tickers can be reused by different securities
        raise ValueError(f"{sym_root} maps to several PERMNOs: {df['permno'].unique()}")
    date = df['yyyymmdd'].to_numpy(dtype='int64')
    price = df['dlyprc'].to_numpy(dtype='float64')
    volume = df['dlyvol'].to_numpy(dtype='float64')
    return date, price, volume

In [3]:
#####################################################################
# We will create and inspect an hdf5 store.                         #
# Note that pd.HDFStore is not appropriate to use for file access   #
# once the store has been written using h5py methods.               #
#####################################################################

# Write
def w(sym_root):

    mode = 'a' if os.path.exists(filename) else 'w'

    date, price, volume = data(sym_root)

    with h5py.File(filename, mode) as fh:

        fh.attrs['start'] = START
        fh.attrs['end'] = END

        path = f"equities/{sym_root}"
        if path in fh:                # re-running replaces the symbol's data
            del fh[path]
        grp = fh.create_group(path)

        ds_date = grp.create_dataset('date', data=date)
        ds_date.attrs['format'] = 'YYYYMMDD'

        ds_price = grp.create_dataset('price',
                                      data=price,
                                      chunks=True,
                                      compression='gzip',
                                      compression_opts=4)
        ds_price.attrs['units'] = 'USD (unadjusted)'
        ds_price.attrs['source'] = source

        ds_volume = grp.create_dataset('volume',
                                       data=volume,
                                       chunks=True,
                                       compression='gzip',
                                       compression_opts=4)
        ds_volume.attrs['units'] = 'shares (unadjusted)'
        ds_volume.attrs['source'] = source

In [4]:
# Inspect
def i(sym_root):
    with h5py.File(filename, 'r') as fh:
        fh.visit(print)
        print(fh[f"equities/{sym_root}/price"].attrs['units'])

In [5]:
# Peek (only reads in necessary chunk)
def p(sym_root, ind):
    with h5py.File(filename, 'r') as fh:
        first_few = fh[f"equities/{sym_root}/price"][0:ind]
    return first_few

In [6]:
# Full read
# Returns numpy ndarrays
def r(sym_root):
    with h5py.File(filename, 'r') as fh:
        p = fh[f"equities/{sym_root}/price"][:]
        v = fh[f"equities/{sym_root}/volume"][:]
    return p,v

In [7]:
if __name__ == "__main__":
    # Examples`

    w("WMT")
    w("MCD")

    p_WMT,v_WMT = r("WMT")
    p_MCD,v_MCD = r("MCD")

Loading library list...
Done
Loading library list...
Done


In [8]:
f = h5py.File("price_volume.h5", 'r')

In [9]:
f.visit(print)

equities
equities/MCD
equities/MCD/date
equities/MCD/price
equities/MCD/volume
equities/WMT
equities/WMT/date
equities/WMT/price
equities/WMT/volume


In [10]:
p = f["equities/MCD/price"]

In [13]:
v = f['equities/MCD/volume']

In [11]:
p

<HDF5 dataset "price": shape (252,), type "<f8">

In [12]:
p[:10]

array([297.04, 294.39, 291.74, 288.99, 291.9 , 290.87, 294.09, 294.15,
       293.47, 291.09])

In [14]:
v[:]

array([ 4458519.,  3114788.,  4615370.,  3407310.,  3097218.,  2176451.,
        2199689.,  2047352.,  1714818.,  2859014.,  2008289.,  2221156.,
        4250885.,  3250224.,  1873976.,  3209675.,  3219625.,  4210521.,
        3899706.,  2524205.,  3476264.,  3380568.,  3971607.,  7581895.,
        5928192.,  3954659.,  4605245.,  3323000.,  2392314.,  2756190.,
        2139326.,  2401617.,  2837573.,  2263994.,  2076811.,  3146598.,
        2199138.,  2595344.,  2811237.,  2507286.,  3601848.,  3028869.,
        2832011.,  2969609.,  2347965.,  2237730.,  2482336.,  2893336.,
        2233076.,  5441262.,  4659308.,  5071675.,  3319435.,  4055380.,
        2867302.,  3653332.,  2556320.,  2985467.,  2572061.,  2604991.,
        3454073.,  2408799.,  2907899.,  2747979.,  4095144.,  6816035.,
        3753206.,  3935142.,  3817458.,  2519172.,  2708630.,  3686881.,
        2912853.,  3383099.,  3402674.,  3017097.,  3129435.,  3275678.,
        4071571.,  4179755.,  3636458.,  3187813., 

In [25]:
dict(f['equities'].attrs)

{}

In [24]:
dict(f['equities/MCD'].attrs)

{}

In [26]:
r('WMT')

(array([159.29, 159.3 , 157.76, 156.71, 158.25, 159.31, 161.29, 161.11,
        161.32, 161.83, 161.07, 163.24, 162.38, 162.39, 162.8 , 160.5 ,
        162.84, 164.27, 165.04, 165.59, 165.25, 168.31, 169.57, 168.66,
        169.81, 169.38, 169.37, 169.28, 170.3 , 169.14, 168.6 , 169.29,
        170.36, 175.86, 173.7 , 175.41, 175.56,  59.6 ,  59.59,  59.62,
         58.61,  58.76,  59.3 ,  60.04,  60.57,  60.36,  60.12,  60.66,
         61.41,  61.28,  61.02,  60.68,  60.86,  60.87,  61.25,  61.45,
         60.87,  60.57,  60.51,  60.72,  60.17,  60.  ,  59.16,  59.43,
         59.5 ,  59.85,  59.78,  59.79,  60.62,  60.09,  60.14,  59.93,
         59.84,  59.65,  59.26,  59.53,  60.14,  59.09,  59.87,  60.21,
         60.16,  60.24,  59.35,  58.85,  59.71,  59.82,  59.87,  60.62,
         60.3 ,  60.44,  60.48,  60.41,  59.86,  59.83,  64.01,  64.65,
         64.18,  65.15,  65.25,  64.84,  65.38,  65.04,  64.92,  64.89,
         65.76,  65.82,  66.6 ,  67.09,  67.15,  65.88,  66.96, 